In [1]:
import gc, sys
from pathlib import Path
import numpy as np
import pandas as pd
from itertools import combinations
import torch
from google.colab import drive, runtime

IN_COLAB = True
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Colab Notebooks/IV Project-Apr 1, 2026')
sys.path.insert(0, str(ROOT))

from src.paths import CLEAN_DATA_V2
from src.benchmark import analytic_benchmark
from src.helper import make_run_dir
from src.fully_connected_colab import (
    compute_batch_size,
    detect_device,
    prepare_gpu_data,
    save_colab_run,
    train_feature_sweep,
)

Mounted at /content/drive


## Load data

In [2]:
DATA_SET = 'rand_D'

df_train = pd.read_parquet(CLEAN_DATA_V2 / (DATA_SET + '_train_v2.parquet'))
df_val   = pd.read_parquet(CLEAN_DATA_V2 / (DATA_SET + '_val_v2.parquet'))
df_test  = pd.read_parquet(CLEAN_DATA_V2 / (DATA_SET + '_test_v2.parquet'))


OUTPUT_DIR = make_run_dir()

## GPU auto-detect

In [3]:
CFG = detect_device()
DEVICE = CFG['DEVICE']
AMP_DTYPE = CFG['POLICY']
USE_AMP = (AMP_DTYPE != torch.float32)

L4  |  VRAM: 24 GB  |  MAX_BATCH=32,768  |  dtype=torch.bfloat16


## Hyperparameters

In [4]:
SEED = 42
MAX_EPOCHS = 100
PATIENCE = 30
LR_PATIENCE = 8
LR_FACTOR = 0.3
WARMUP_EPOCHS = 5
NEURONS = 80
HIDDEN_LAYERS = 3

BATCH_SIZE = compute_batch_size(len(df_train), CFG['MAX_BATCH'])
BASE_LR = 1e-3
BASE_BATCH = 4096
INIT_LR = BASE_LR * (BATCH_SIZE / BASE_BATCH) ** 0.5

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

steps_per_epoch = len(df_train) // BATCH_SIZE
print(f'MAX_BATCH={CFG["MAX_BATCH"]:,}  adaptive BATCH_SIZE={BATCH_SIZE:,}  '
      f'INIT_LR={INIT_LR:.6f}  n_train={len(df_train):,}  '
      f'steps/epoch~{steps_per_epoch}')
print(f'MAX_EPOCHS={MAX_EPOCHS}  PATIENCE={PATIENCE}  '
      f'WARMUP={WARMUP_EPOCHS} epochs')

MAX_BATCH=32,768  adaptive BATCH_SIZE=16,384  INIT_LR=0.002000  n_train=843,153  steps/epoch~51
MAX_EPOCHS=100  PATIENCE=30  WARMUP=5 epochs


## Feature definitions

In [5]:
from itertools import combinations

FEATURES_3 = ['delta', 'T', 'spy_ret']
EXTRA_FEATURES = ['vix_lag', 'vix_mom_lag', 'vix_mom', 'gamma', 'theta', 'vega', 'rho']
ALL_FEATURES = FEATURES_3 + EXTRA_FEATURES
TARGET = 'd_iv'

# Generate ALL possible combinations of EXTRA_FEATURES (0 to 7)
feature_combos = []
base_name = '+'.join(FEATURES_3)
feature_combos.append((base_name, FEATURES_3.copy()))  # Base case

for r in range(1, len(EXTRA_FEATURES) + 1):
    for combo in combinations(EXTRA_FEATURES, r):
        feats = FEATURES_3 + list(combo)
        name = '+'.join(feats)
        feature_combos.append((name, feats))

# Calculate breakdown for all sizes
n_plus_0 = 1
n_plus_1 = len(list(combinations(EXTRA_FEATURES, 1)))
n_plus_2 = len(list(combinations(EXTRA_FEATURES, 2)))
n_plus_3 = len(list(combinations(EXTRA_FEATURES, 3)))
n_plus_4 = len(list(combinations(EXTRA_FEATURES, 4)))
n_plus_5 = len(list(combinations(EXTRA_FEATURES, 5)))
n_plus_6 = len(list(combinations(EXTRA_FEATURES, 6)))
n_plus_7 = len(list(combinations(EXTRA_FEATURES, 7)))

print(f'Total feature combinations: {len(feature_combos)}')
print('  Base (3F):  1')
print(f'  3F + 1:     {n_plus_1}')
print(f'  3F + 2:     {n_plus_2}')
print(f'  3F + 3:     {n_plus_3}')
print(f'  3F + 4:     {n_plus_4}')
print(f'  3F + 5:     {n_plus_5}')
print(f'  3F + 6:     {n_plus_6}')
print(f'  3F + 7:     {n_plus_7}')

Total feature combinations: 128
  Base (3F):  1
  3F + 1:     7
  3F + 2:     21
  3F + 3:     35
  3F + 4:     35
  3F + 5:     21
  3F + 6:     7
  3F + 7:     1


## Pre-allocate on GPU

In [6]:
gpu_data = prepare_gpu_data(df_train, df_val, df_test, ALL_FEATURES, TARGET, DEVICE)
COL_IDX = gpu_data['col_idx']

Data on GPU  |  VRAM used: 0.26 GB / 24 GB  |  Free: 23.4 GB
Train: 843,153  Val: 240,901  Test: 120,451  Features: 10


## Analytic benchmark

In [7]:
hw = analytic_benchmark(df_train, df_val, df_test, target=TARGET)

Analytic Benchmark
SSE = 8.3745  RMSE = 0.008338
Coefficients: a = -0.165665, b = -0.000516, c = 0.032673


## Train all feature combinations

In [8]:
train_kw = dict(
    Xtr=gpu_data['Xtr'],
    Xva=gpu_data['Xva'],
    Xte=gpu_data['Xte'],
    ytr=gpu_data['ytr'],
    yva=gpu_data['yva'],
    y_test=gpu_data['y_test'],
    hw_sse=hw['sse'],
    all_feature_names=ALL_FEATURES,
    device=DEVICE,
    amp_dtype=AMP_DTYPE,
    use_amp=USE_AMP,
    nan_mask_tr=gpu_data['nan_mask_tr'],
    nan_mask_va=gpu_data['nan_mask_va'],
    nan_mask_ytr=gpu_data['nan_mask_ytr'],
    nan_mask_yva=gpu_data['nan_mask_yva'],
    seed=SEED,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    lr_patience=LR_PATIENCE,
    lr_factor=LR_FACTOR,
    init_lr=INIT_LR,
    warmup_epochs=WARMUP_EPOCHS,
    neurons=NEURONS,
    hidden_layers=HIDDEN_LAYERS,
)

sep = "=" * 70
print('\n' + sep)
print(f'  TRAINING {len(feature_combos)} MODELS')
print(f'  GPU: {CFG["GPU"]}  |  Batch: {BATCH_SIZE:,}  |  AMP: {USE_AMP}  |  Max epochs: {MAX_EPOCHS}')
print(sep + '\n')

all_results, elapsed_s = train_feature_sweep(
    feature_combos,
    col_idx=COL_IDX,
    train_kwargs=train_kw,
    print_every=25,
)

print(f'\nDone: {elapsed_s / 60:.1f} min for {len(all_results)} models '
      f'(avg {elapsed_s / max(len(all_results), 1):.1f}s/model)')


  TRAINING 128 MODELS
  GPU: L4  |  Batch: 16,384  |  AMP: True  |  Max epochs: 100

  [  1/128] delta+T+spy_ret                SSE=10.9756  Gain=-31.1%  ep=100  19.1s  elapsed=0.3min
  [ 25/128] delta+T+spy_ret+gamma+vega     SSE=15.1161  Gain=-80.5%  ep=100  12.3s  elapsed=5.3min
  [ 50/128] delta+T+spy_ret+vix_mom_lag+gamma+vega SSE=11.6591  Gain=-39.2%  ep=100  12.5s  elapsed=10.5min
  [ 75/128] delta+T+spy_ret+vix_lag+vix_mom+gamma+theta SSE=9.3742  Gain=-11.9%  ep=100  12.5s  elapsed=15.7min
  [100/128] delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+gamma+theta SSE=9.3099  Gain=-11.2%  ep=100  12.3s  elapsed=21.0min
  [125/128] delta+T+spy_ret+vix_lag+vix_mom_lag+gamma+theta+vega+rho SSE=14.7164  Gain=-75.7%  ep=100  12.2s  elapsed=26.2min
  [128/128] delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+gamma+theta+vega+rho SSE=9.4019  Gain=-12.3%  ep=100  12.4s  elapsed=26.8min

Done: 26.8 min for 128 models (avg 12.6s/model)


## Results summary

In [9]:
summary, df_results = save_colab_run(
    OUTPUT_DIR,
    y_test=gpu_data['y_test'],
    hw=hw,
    models=all_results,
)
summary

,Model,SSE,MSE,RMSE,MAE,MeanError,MedianAE,R2,Training_time,Gain_vs_Analytic,Gain_Incremental
0,Analytic,8.374537,0.000070,0.008338,0.003523,0.000748,0.001885,0.125011,None,None,None
1,delta+T+spy_ret,10.975561,0.000091,0.009546,0.005470,-0.000109,0.003735,-0.146749,13.5s,-31.06%,None
2,delta+T+spy_ret+vix_lag,20.193176,0.000168,0.012948,0.008821,-0.000484,0.006409,-1.109824,12.3s,-141.13%,-83.98%
3,delta+T+spy_ret+vix_mom_lag,20.058388,0.000167,0.012905,0.008799,-0.000897,0.006396,-1.095741,12.5s,-139.52%,0.67%
4,delta+T+spy_ret+vix_mom,20.851032,0.000173,0.013157,0.008801,0.000345,0.006275,-1.178558,12.5s,-148.98%,-3.95%
...,...,...,...,...,...,...,...,...,...,...,...
125,delta+T+spy_ret+vix_lag+vix_mom_lag+gamma+thet...,14.716359,0.000122,0.011053,0.007687,-0.001642,0.005814,-0.537595,12.2s,-75.73%,-4.91%
126,delta+T+spy_ret+vix_lag+vix_mom+gamma+theta+ve...,13.029432,0.000108,0.010401,0.007021,-0.000123,0.005172,-0.361342,12.2s,-55.58%,11.46%
127,delta+T+spy_ret+vix_mom_lag+vix_mom+gamma+thet...,12.411659,0.000103,0.010151,0.006886,-0.000777,0.005107,-0.296795,12.2s,-48.21%,4.74%
128,delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+ga...,9.401947,0.000078,0.008835,0.005696,-0.000421,0.004146,0.017665,12.4s,-12.27%,24.25%


## Top 10 by Gain vs Hull-White

In [10]:
print('Top 10 feature combos by Gain vs Hull-White:\n')
top10 = df_results.head(10)[['combo_name', 'n_features', 'SSE', 'RMSE', 'Gain_vs_HW_%', 'training_time_s']].copy()
top10['Gain_vs_HW_%'] = top10['Gain_vs_HW_%'].round(2)
top10['RMSE'] = top10['RMSE'].round(6)
top10['SSE'] = top10['SSE'].round(4)
top10['training_time_s'] = top10['training_time_s'].round(1)
top10

Top 10 feature combos by Gain vs Hull-White:



,combo_name,n_features,SSE,RMSE,Gain_vs_HW_%,training_time_s
0,delta+T+spy_ret+vix_lag+vix_mom+vega+rho,7,8.4444,0.008373,-0.83,12.3
1,delta+T+spy_ret+vix_lag+vix_mom+theta+rho,7,8.7442,0.008520,-4.41,12.7
2,delta+T+spy_ret+vix_mom_lag+vix_mom+vega+rho,7,8.7822,0.008539,-4.87,12.7
3,delta+T+spy_ret+vix_mom_lag+vix_mom+theta+vega,7,8.8154,0.008555,-5.26,12.7
4,delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+ga...,8,9.3099,0.008792,-11.17,12.3
5,delta+T+spy_ret+vix_mom+theta+vega+rho,7,9.3139,0.008793,-11.22,12.3
6,delta+T+spy_ret+vix_mom+gamma+theta+rho,7,9.3308,0.008801,-11.42,12.4
7,delta+T+spy_ret+vix_lag+vix_mom+gamma+theta,7,9.3742,0.008822,-11.94,12.4
8,delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+ga...,10,9.4019,0.008835,-12.27,12.4
9,delta+T+spy_ret+vix_lag+vix_mom+theta+vega,7,9.4520,0.008858,-12.87,12.3


## Best per group (3F, +1, +2, +3)

In [11]:
for nf_label, n in [('3F (base)', 3), ('+1 (4F)', 4), ('+2 (5F)', 5), ('+3 (6F)', 6)]:
    sub = df_results[df_results['n_features'] == n]
    if len(sub) == 0:
        continue
    best = sub.iloc[0]
    print(f"{nf_label}: {best['combo_name']}")
    print(f"    SSE={best['SSE']:.4f}  RMSE={best['RMSE']:.6f}  Gain={best['Gain_vs_HW_%']:.2f}%\n")

3F (base): delta+T+spy_ret
    SSE=10.9756  RMSE=0.009546  Gain=-31.06%

+1 (4F): delta+T+spy_ret+gamma
    SSE=14.9572  RMSE=0.011143  Gain=-78.60%

+2 (5F): delta+T+spy_ret+vix_mom+theta
    SSE=10.6585  RMSE=0.009407  Gain=-27.27%

+3 (6F): delta+T+spy_ret+vix_mom+gamma+theta
    SSE=10.2516  RMSE=0.009226  Gain=-22.41%



## Summary statistics

In [12]:
total_time_s = df_results['training_time_s'].sum()
print(f'Total training time: {total_time_s / 60:.1f} min ({total_time_s / 3600:.2f} hr)')
print(f'Models trained: {len(df_results)}')
print(f'Best overall: {df_results.iloc[0]["combo_name"]} (Gain={df_results.iloc[0]["Gain_vs_HW_%"]:.2f}%)')

Total training time: 26.7 min (0.44 hr)
Models trained: 128
Best overall: delta+T+spy_ret+vix_lag+vix_mom+vega+rho (Gain=-0.83%)


## Cleanup

In [13]:
del all_results, gpu_data
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f'Final VRAM: {(total - free) / 1e9:.2f} GB / {total / 1e9:.0f} GB')

print(f'Total training time: {total_time_s / 60:.1f} min for {len(df_results)} models')

if IN_COLAB:
    runtime.unassign()

Final VRAM: 0.36 GB / 24 GB
Total training time: 26.7 min for 128 models
